In [4]:
# ==============================================================================
# IBM Z DATATHON - COMPLETE 3D SMART TRAFFIC SYSTEM (SINGLE-CELL ALL-IN-ONE)
# Works on macOS Jupyter, JupyterLite (Python 3.14) & IBM Z without nbformat
# ==============================================================================
import sqlite3
import json
import datetime
import html
from IPython.display import HTML, Markdown, IFrame, display

# 1. SQLITE DATABASE & ANALYTICS ENGINE
class SmartTrafficSystem:
    def __init__(self):
        self.conn = sqlite3.connect(":memory:", check_same_thread=False)
        self.conn.row_factory = sqlite3.Row
        self._init_db()

    def _init_db(self):
        cur = self.conn.cursor()
        cur.execute("CREATE TABLE segments (id TEXT PRIMARY KEY, name TEXT, junc TEXT, veh INT, density REAL, level TEXT, speed REAL, queue REAL, dur REAL, entered INT, exited INT, status TEXT, prov TEXT)")
        cur.execute("CREATE TABLE signals (id TEXT PRIMARY KEY, name TEXT, intended TEXT, observed TEXT, fault INT, override INT, health TEXT)")
        cur.execute("CREATE TABLE corridors (id TEXT PRIMARY KEY, veh_id TEXT, vtype TEXT, route TEXT, status TEXT, act_sec REAL, blk_det_sec REAL, path_clr_sec REAL, full_clr_sec REAL, saved_sec REAL)")
        cur.execute("CREATE TABLE env_metrics (seg_id TEXT PRIMARY KEY, co2_base REAL, co2_opt REAL, co2_saved REAL, fuel_base REAL, fuel_opt REAL, fuel_saved REAL, model_tag TEXT)")
        cur.execute("CREATE TABLE class_counts (vclass TEXT PRIMARY KEY, cnt INT, avg_spd REAL, ef_g_km REAL, idle_l_hr REAL)")

        cur.executemany("INSERT INTO segments VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)", [
            ("SEG-N1", "North Hospital Arterial", "INT-NORTH", 42, 23.3, "MODERATE", 38.5, 45.0, 4.2, 4820, 4795, "ONLINE", "SIMULATED_LIVE"),
            ("SEG-C1", "Central Civic Junction", "INT-CENTER", 68, 45.3, "HIGH", 19.4, 115.0, 11.5, 6390, 6310, "ONLINE", "SIMULATED_LIVE"),
            ("SEG-S1", "South Industrial Ring", "INT-SOUTH", 54, 25.7, "MODERATE", 33.2, 62.0, 6.0, 5140, 5110, "ONLINE", "SIMULATED_LIVE"),
            ("SEG-E1", "East Tech Expressway", "INT-EAST", 29, 18.1, "LOW", 51.0, 18.0, 0.0, 3980, 3972, "ONLINE", "SIMULATED_LIVE")
        ])
        cur.executemany("INSERT INTO signals VALUES (?,?,?,?,?,?,?)", [
            ("INT-NORTH", "North Medical Gate", "GREEN", "GREEN", 0, 1, "HEALTHY (99.9%)"),
            ("INT-CENTER", "Central Plaza 4-Way", "GREEN", "GREEN", 0, 1, "HEALTHY (99.8%)"),
            ("INT-SOUTH", "South Cargo Ring", "GREEN", "RED", 1, 0, "FAULT: DRONE CV MISMATCH"),
            ("INT-EAST", "East Tech Link", "RED", "RED", 0, 0, "HEALTHY (99.9%)")
        ])
        cur.executemany("INSERT INTO corridors VALUES (?,?,?,?,?,?,?,?,?,?)", [
            ("GC-01", "AMB-101", "Ambulance", "SEG-N1 -> SEG-C1 -> SEG-S1", "ACTIVE_CORRIDOR", 1.8, 2.4, 16.3, 52.0, 118.0),
            ("GC-02", "FIR-204", "Fire Engine", "SEG-E1 -> SEG-C1", "RESOLVED", 2.1, 0.0, 12.0, 44.0, 94.0),
            ("GC-03", "POL-309", "Police Vehicle", "SEG-N1 -> SEG-E1", "RESOLVED", 1.5, 0.0, 11.0, 39.0, 67.0),
            ("GC-04", "AMB-108", "Ambulance", "SEG-S1 -> SEG-C1", "CLEARED_BLOCKAGE", 1.9, 2.8, 18.2, 58.4, 142.0)
        ])
        cur.executemany("INSERT INTO env_metrics VALUES (?,?,?,?,?,?,?,?)", [
            ("SEG-N1", 412.5, 338.2, 74.3, 178.4, 144.5, 33.9, "MODEL-ESTIMATE (ARAI/IPCC)"),
            ("SEG-C1", 685.0, 524.0, 161.0, 298.0, 224.1, 73.9, "MODEL-ESTIMATE (ARAI/IPCC)"),
            ("SEG-S1", 548.2, 459.1, 89.1, 236.5, 196.8, 39.7, "MODEL-ESTIMATE (ARAI/IPCC)"),
            ("SEG-E1", 289.4, 251.8, 37.6, 124.2, 107.6, 16.6, "MODEL-ESTIMATE (ARAI/IPCC)")
        ])
        cur.executemany("INSERT INTO class_counts VALUES (?,?,?,?,?)", [
            ("Heavy", 2080, 28.4, 780.0, 2.40),
            ("Light", 9140, 39.2, 165.0, 0.85),
            ("Two-Wheeler", 5840, 42.1, 45.0, 0.22),
            ("Three-Wheeler", 2270, 31.0, 85.0, 0.38),
            ("Emergency", 72, 54.6, 240.0, 1.20),
            ("Other", 928, 16.5, 0.0, 0.00)
        ])
        self.conn.commit()

    def get_snapshot(self):
        cur = self.conn.cursor()
        segs = [dict(r) for r in cur.execute("SELECT * FROM segments").fetchall()]
        sigs = [dict(r) for r in cur.execute("SELECT * FROM signals").fetchall()]
        cors = [dict(r) for r in cur.execute("SELECT * FROM corridors").fetchall()]
        envs = [dict(r) for r in cur.execute("SELECT * FROM env_metrics").fetchall()]
        clss = [dict(r) for r in cur.execute("SELECT * FROM class_counts").fetchall()]
        return {
            "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "freshness": "LIVE (<1.0s latency | Simulated Telemetry)",
            "segments": segs,
            "signals": sigs,
            "corridors": cors,
            "env": envs,
            "classes": clss,
            "kpis": {
                "em_detected": 14,
                "em_passed": 13,
                "success_rate": 92.9,
                "act_time_sec": 1.82,
                "blk_detect_sec": 2.45,
                "path_clr_sec": 16.35,
                "full_clr_sec": 54.67,
                "total_entered": sum(s["entered"] for s in segs),
                "total_exited": sum(s["exited"] for s in segs),
                "co2_opt_kg": round(sum(e["co2_opt"] for e in envs), 1),
                "co2_saved_kg": round(sum(e["co2_saved"] for e in envs), 1),
                "fuel_saved_l": round(sum(e["fuel_saved"] for e in envs), 1)
            }
        }

traffic_sys = SmartTrafficSystem()
snap_json = json.dumps(traffic_sys.get_snapshot())

# 2. 3D WEBGL / CANVAS DIGITAL TWIN + 3D SURFACE PLUME + COPILOT DASHBOARD
ui_lines = [
    "<!DOCTYPE html><html><head><meta charset='UTF-8'><style>",
    "*{box-sizing:border-box;margin:0;padding:0;font-family:-apple-system,BlinkMacSystemFont,sans-serif;}",
    "body{background:#060a12;color:#e2e8f0;height:100vh;display:flex;flex-direction:column;overflow:hidden;}",
    "#hdr{background:#0f172a;border-bottom:1px solid #334155;padding:7px 12px;display:flex;justify-content:space-between;align-items:center;}",
    ".badge{background:#2563eb;color:#fff;padding:2px 7px;border-radius:4px;font-size:11px;font-weight:700;}",
    ".pill{background:rgba(16,185,129,0.15);border:1px solid #10b981;color:#34d399;padding:2px 8px;border-radius:99px;font-size:11px;}",
    "#sbar{background:#09101d;border-bottom:1px solid #1e293b;padding:5px 10px;display:flex;flex-wrap:wrap;gap:4px;}",
    ".sbtn{background:#1e293b;color:#cbd5e1;border:1px solid #334155;border-radius:4px;padding:4px 7px;font-size:10.5px;cursor:pointer;font-weight:600;}",
    ".sbtn:hover{background:#334155;color:#fff;} .sbtn.act{background:#2563eb;color:#fff;border-color:#60a5fa;}",
    ".sbtn.em{border-color:#ef4444;color:#fca5a5;} .sbtn.em.act{background:#dc2626;color:#fff;}",
    "#grid{display:grid;grid-template-columns:285px 1fr 310px;flex:1;min-height:0;}",
    ".pnl{background:#0b1120;border-right:1px solid #1e293b;padding:8px;overflow-y:auto;display:flex;flex-direction:column;gap:7px;}",
    ".pnl.rt{border-right:none;border-left:1px solid #1e293b;}",
    ".crd{background:#0f172a;border:1px solid #1e293b;border-radius:7px;padding:7px 9px;}",
    ".ctit{font-size:10.5px;font-weight:700;text-transform:uppercase;color:#94a3b8;margin-bottom:5px;display:flex;justify-content:space-between;}",
    ".kgrid{display:grid;grid-template-columns:1fr 1fr;gap:5px;}",
    ".kbox{background:#1e293b;padding:5px 7px;border-radius:5px;}",
    ".kval{font-size:14px;font-weight:700;color:#38bdf8;} .klbl{font-size:9.5px;color:#94a3b8;}",
    "#vwrap{position:relative;background:#04070e;width:100%;height:100%;overflow:hidden;}",
    "#c3d{width:100%;height:100%;display:block;cursor:grab;}",
    "#cam-bar{position:absolute;top:8px;left:8px;display:flex;flex-wrap:wrap;gap:5px;z-index:5;}",
    ".cbtn{background:rgba(15,23,42,0.88);border:1px solid #475569;color:#e2e8f0;padding:4px 8px;font-size:10.5px;border-radius:5px;cursor:pointer;}",
    ".cbtn.act{background:#0284c7;border-color:#38bdf8;color:#fff;}",
    "#hud{position:absolute;bottom:8px;left:8px;right:8px;background:rgba(15,23,42,0.9);border:1px solid #334155;border-radius:6px;padding:6px 10px;display:flex;justify-content:space-between;font-size:10.5px;}",
    "table{width:100%;border-collapse:collapse;font-size:10px;} th,td{padding:3px 4px;text-align:left;border-bottom:1px solid #1e293b;} th{color:#64748b;}",
    "#chat{flex:1;background:#070b14;border:1px solid #1e293b;border-radius:5px;padding:6px;overflow-y:auto;display:flex;flex-direction:column;gap:5px;min-height:150px;max-height:220px;}",
    ".m{padding:5px 7px;border-radius:5px;font-size:10.5px;line-height:1.35;}",
    ".m.u{background:#1e3a8a;color:#e0f2fe;align-self:flex-end;max-width:90%;}",
    ".m.b{background:#1e293b;color:#f1f5f9;align-self:flex-start;border-left:3px solid #38bdf8;max-width:95%;}",
    ".chips{display:flex;flex-wrap:wrap;gap:3px;margin-top:4px;}",
    ".chip{background:#1e293b;border:1px solid #334155;color:#93c5fd;font-size:9.5px;padding:2px 5px;border-radius:3px;cursor:pointer;}",
    ".chip:hover{background:#2563eb;color:#fff;}",
    ".crow{display:flex;gap:4px;margin-top:5px;} .crow input{flex:1;background:#0f172a;border:1px solid #334155;color:#fff;padding:4px 7px;border-radius:4px;font-size:10.5px;}",
    ".crow button{background:#2563eb;border:none;color:#fff;padding:4px 9px;border-radius:4px;font-size:10.5px;cursor:pointer;font-weight:600;}",
    "</style></head><body>",

    "<div id='hdr'><div><span class='badge'>IBM Z DATATHON</span> <b style='font-size:13px;margin-left:6px;'>3D AI Smart Traffic Management, Green Corridor & GenAI Copilot</b></div>",
    "<div style='display:flex;gap:8px;align-items:center;'><span id='pbadge' class='pill'>● LIVE 3D DIGITAL TWIN (60 FPS)</span><span id='clk' style='font-size:11px;color:#94a3b8;font-family:monospace;'></span></div></div>",

    "<div id='sbar'>",
    "<button class='sbtn act' onclick='setScen(1,this)'>1. Normal Traffic</button>",
    "<button class='sbtn em' onclick='setScen(2,this)'>2. Ambulance Green Corridor</button>",
    "<button class='sbtn em' onclick='setScen(3,this)'>3. Ambulance Stuck (Verify & Clear)</button>",
    "<button class='sbtn' onclick='setScen(4,this)'>4. Signal Fault Alert</button>",
    "<button class='sbtn' onclick='setScen(5,this)'>5. Drone CV Monitoring</button>",
    "<button class='sbtn' onclick='setScen(6,this)'>6. Volume Analytics</button>",
    "<button class='sbtn' onclick='setScen(7,this)'>7. CO2 & Fuel Analytics</button>",
    "<button class='sbtn' onclick='setScen(8,this)'>8. GenAI Copilot Query</button>",
    "<button class='sbtn' onclick='setScen(9,this)'>9. Sensor Outage Fallback</button>",
    "<button class='sbtn' onclick='setScen(10,this)'>10. Daily Audit Report</button>",
    "</div>",

    "<div id='grid'>",
    "<div class='pnl'>",
    "<div class='crd'><div class='ctit'><span>Emergency Green Corridor</span><span id='gc-st' style='color:#10b981;'>● ACTIVE</span></div>",
    "<div class='kgrid'>",
    "<div class='kbox'><div class='kval' id='kv-pass'>13 / 14</div><div class='klbl'>Passed (92.9% Success)</div></div>",
    "<div class='kbox'><div class='kval'>1.82s</div><div class='klbl'>Avg Corridor Activation</div></div>",
    "<div class='kbox'><div class='kval'>2.45s</div><div class='klbl'>Blockage Detect Time</div></div>",
    "<div class='kbox'><div class='kval' id='kv-clr'>16.3s / 54.7s</div><div class='klbl'>Path / Full Queue Clr</div></div>",
    "</div>",
    "<div id='amb-box' style='margin-top:5px;font-size:10px;background:#1e293b;padding:5px;border-radius:4px;border-left:3px solid #10b981;'>",
    "<b>AMB-101:</b> <span id='amb-st'>Green Corridor Active</span><br><b>Spd:</b> <span id='amb-sp'>54.5 km/h</span> | <b>Check:</b> <span id='amb-vf'>Moving Freely</span></div></div>",

    "<div class='crd'><div class='ctit'><span>Monitored Road Segments</span></div><table><thead><tr><th>Seg</th><th>Veh</th><th>Spd</th><th>Queue</th><th>State</th></tr></thead><tbody id='tb-seg'></tbody></table></div>",
    "<div class='crd'><div class='ctit'><span>Signals & Drone CV Audit</span></div><table><thead><tr><th>Junc</th><th>Ctrl</th><th>Drone</th><th>Status</th></tr></thead><tbody id='tb-sig'></tbody></table></div>",
    "<div class='crd'><div class='ctit'><span>6-Class CV Vehicle Counts</span></div><div id='cls-box' style='display:grid;grid-template-columns:1fr 1fr;gap:3px;font-size:10px;'></div></div>",
    "</div>",

    "<div id='vwrap'>",
    "<div id='cam-bar'>",
    "<button class='cbtn act' onclick='setCam(0,this)'>🎥 3D City Orbit (Drag to Rotate)</button>",
    "<button class='cbtn' onclick='setCam(1,this)'>🛸 Drone Top-Down 3D</button>",
    "<button class='cbtn' onclick='setCam(2,this)'>🚑 Ambulance Chase 3D</button>",
    "<button class='cbtn' onclick='setCam(3,this)'>🌋 Toggle 3D Congestion Plume Mesh</button>",
    "</div>",
    "<canvas id='c3d'></canvas>",
    "<div id='hud'>",
    "<div><span style='color:#ef4444;'>■</span> Emergency <span style='color:#3b82f6;margin-left:5px;'>■</span> Heavy <span style='color:#22d3ee;margin-left:5px;'>■</span> Light <span style='color:#f97316;margin-left:5px;'>■</span> 2W <span style='color:#eab308;margin-left:5px;'>■</span> 3W <span style='color:#a855f7;margin-left:5px;'>■</span> Other <span style='color:#10b981;margin-left:5px;'>■</span> 3D Green Corridor</div>",
    "<div id='sbanner' style='color:#38bdf8;font-weight:700;'>Scenario 1: Normal 6-Class CV Tracking</div>",
    "</div></div>",

    "<div class='pnl rt'>",
    "<div class='crd'><div class='ctit'><span>CO2 & Fuel Analytics</span><span style='color:#f59e0b;font-size:9px;'>MODEL ESTIMATE</span></div>",
    "<div class='kgrid'>",
    "<div class='kbox'><div class='kval' style='color:#34d399;' id='kv-co2'>-362.0 kg</div><div class='klbl'>CO2 Saved Today</div></div>",
    "<div class='kbox'><div class='kval' style='color:#fbbf24;' id='kv-fuel'>164.1 L</div><div class='klbl'>Idling Fuel Saved</div></div>",
    "</div><div style='font-size:9.5px;color:#94a3b8;margin-top:4px;'>Optimized CO2: <b id='kv-tot' style='color:#fff;'>1573.1 kg</b> (vs 1935.1 kg Base) | Delay: <b style='color:#34d399;'>-24.6%</b></div></div>",

    "<div class='crd'><div class='ctit'><span>Incident & Clearance Log</span></div><div id='inc-log' style='font-size:9.5px;max-height:85px;overflow-y:auto;display:flex;flex-direction:column;gap:3px;'></div></div>",

    "<div class='crd' style='flex:1;display:flex;flex-direction:column;'><div class='ctit'><span>🤖 GenAI Traffic Copilot</span><span style='color:#38bdf8;font-size:9px;'>RBAC + SQL RAG</span></div>",
    "<div id='chat'><div class='m b'><b>GenAI Traffic Copilot Active.</b> Connected to live 3D telemetry & SQLite DB. Click a prompt chip or ask a question below.</div></div>",
    "<div class='chips'>",
    "<span class='chip' onclick='askBot(\"How many ambulances passed through the green corridor today?\")'>Ambulances today?</span>",
    "<span class='chip' onclick='askBot(\"Is any ambulance currently stuck in traffic?\")'>Ambulance stuck?</span>",
    "<span class='chip' onclick='askBot(\"Which road has the highest congestion?\")'>Highest congestion?</span>",
    "<span class='chip' onclick='askBot(\"How much fuel and CO2 were saved today?\")'>CO2 & Fuel saved?</span>",
    "<span class='chip' onclick='askBot(\"Which traffic signal is experiencing a problem?\")'>Signal fault?</span>",
    "<span class='chip' onclick='askBot(\"Force change signal INT-CENTER to green now\")'>Test Safety Lock</span>",
    "</div>",
    "<div class='crow'><input id='cinp' placeholder='Ask Copilot...' onkeydown='if(event.key===\"Enter\")sendChat()'/><button onclick='sendChat()'>Ask</button></div>",
    "</div></div></div>",

    "<script>",
    "const DATA = " + snap_json + ";",
    "let t = 0, showPlume = false, yaw = 0.68, pitch = 0.55, zoom = 4.2, dragging = false, lx = 0, ly = 0;",
    "let gcActive = true, stuckMode = false, stopTimer = 0, blkVerified = false, clrTimer = 0, fallbackMode = false;",
    "let co2Saved = DATA.kpis.co2_saved_kg, co2Tot = DATA.kpis.co2_opt_kg, fuelSaved = DATA.kpis.fuel_saved_l, emPassed = DATA.kpis.em_passed;",
    "const clsMap = {Heavy:2080, Light:9140, 'Two-Wheeler':5840, 'Three-Wheeler':2270, Emergency:72, Other:928};",
    "const sigs = DATA.signals;",

    "const cv = document.getElementById('c3d'), ctx = cv.getContext('2d');",
    "function resize(){ cv.width = cv.parentElement.clientWidth; cv.height = cv.parentElement.clientHeight; }",
    "window.addEventListener('resize', resize); setTimeout(resize, 30);",
    "cv.addEventListener('mousedown', e => { dragging=true; lx=e.clientX; ly=e.clientY; });",
    "window.addEventListener('mouseup', () => dragging=false);",
    "window.addEventListener('mousemove', e => { if(!dragging)return; yaw += (e.clientX-lx)*0.008; pitch = Math.max(0.2, Math.min(1.45, pitch+(e.clientY-ly)*0.008)); lx=e.clientX; ly=e.clientY; });",
    "cv.addEventListener('wheel', e => { e.preventDefault(); zoom = Math.max(2.2, Math.min(7.5, zoom - e.deltaY*0.003)); }, {passive:false});",

    "function proj(x, y, z){",
    "  const rx = x*Math.cos(yaw) - z*Math.sin(yaw);",
    "  const rz = x*Math.sin(yaw) + z*Math.cos(yaw);",
    "  const ry = y*Math.cos(pitch) - rz*Math.sin(pitch);",
    "  const d = y*Math.sin(pitch) + rz*Math.cos(pitch) + 210;",
    "  const s = (165 * zoom) / Math.max(25, d);",
    "  return {x: cv.width/2 + rx*s, y: cv.height/2 - ry*s, s: s, d: d};",
    "}",

    "function drawBox3D(x, y, z, w, h, l, colTop, colSide, wireCol){",
    "  const hw = w/2, hl = l/2;",
    "  const pts = [[-hw,0,-hl],[hw,0,-hl],[hw,0,hl],[-hw,0,hl],[-hw,h,-hl],[hw,h,-hl],[hw,h,hl],[-hw,h,hl]].map(p => proj(x+p[0], y+p[1], z+p[2]));",
    "  const faces = [[0,1,5,4,colSide],[1,2,6,5,colSide],[2,3,7,6,colSide],[3,0,4,7,colSide],[4,5,6,7,colTop]];",
    "  faces.forEach(f => {",
    "    ctx.beginPath(); ctx.moveTo(pts[f[0]].x, pts[f[0]].y);",
    "    for(let i=1;i<4;i++) ctx.lineTo(pts[f[i]].x, pts[f[i]].y);",
    "    ctx.closePath(); ctx.fillStyle = f[4]; ctx.fill();",
    "    ctx.strokeStyle = wireCol || 'rgba(56,189,248,0.28)'; ctx.lineWidth = 0.8; ctx.stroke();",
    "  });",
    "  return pts[4];",
    "}",

    "const bldgs = [[-32,-55,16,24,16],[-32,-18,16,32,16],[-32,20,16,26,16],[-32,58,16,20,16],[32,-55,16,22,16],[32,-18,18,38,16],[32,20,16,28,16],[32,58,16,22,16]];",
    "const vehs = [{id:'AMB-101', cls:'Emergency', x:-3.5, z:-85, vx:0, vz:0.62, w:2.4, h:2.4, l:5.2, c1:'#ffffff', c2:'#cbd5e1', blk:false}];",
    "const specs = [",
    "  {cls:'Heavy', w:2.6, h:3.2, l:7.2, c1:'#60a5fa', c2:'#2563eb', sp:0.32},",
    "  {cls:'Light', w:2.0, h:1.5, l:4.0, c1:'#67e8f9', c2:'#0891b2', sp:0.45},",
    "  {cls:'Two-Wheeler', w:0.9, h:1.3, l:2.0, c1:'#fdba74', c2:'#ea580c', sp:0.52},",
    "  {cls:'Three-Wheeler', w:1.5, h:1.6, l:2.8, c1:'#fde047', c2:'#ca8a04', sp:0.36},",
    "  {cls:'Other', w:0.8, h:1.2, l:1.8, c1:'#d8b4fe', c2:'#9333ea', sp:0.22}",
    "];",
    "for(let i=0;i<24;i++){",
    "  const sp = specs[i % specs.length];",
    "  if(i<15){",
    "    const isBlk = i<4;",
    "    vehs.push({id:'TRK-'+(101+i), cls:sp.cls, x:isBlk?-3.5:3.5, z:isBlk?(-22+i*6.5):(-85+i*12), vx:0, vz:isBlk?sp.sp:-sp.sp, w:sp.w, h:sp.h, l:sp.l, c1:sp.c1, c2:sp.c2, blk:isBlk});",
    "  } else {",
    "    const cz = (i%2===0)?-38:38, dir = (i%2===0)?1:-1;",
    "    vehs.push({id:'TRK-'+(101+i), cls:sp.cls, x:-75+(i*13)%150, z:cz+(dir*2.5), vx:dir*sp.sp, vz:0, w:sp.l, h:sp.h, l:sp.w, c1:sp.c1, c2:sp.c2, blk:false});",
    "  }",
    "}",

    "function addLog(tit, msg, col){",
    "  const el = document.getElementById('inc-log'), d = document.createElement('div');",
    "  d.style.cssText = 'background:#1e293b;padding:4px 6px;border-radius:4px;border-left:3px solid '+col+';';",
    "  d.innerHTML = '<b>['+new Date().toLocaleTimeString()+'] '+tit+'</b><br>'+msg;",
    "  el.prepend(d);",
    "}",
    "addLog('INC-901 RESOLVED', 'AMB-108 blockage verified in 2.8s; cleared in 18.2s (Full queue: 54.6s).', '#10b981');",
    "addLog('INC-902 ACTIVE FAULT', 'INT-SOUTH: Controller GREEN vs Drone CV RED mismatch.', '#ef4444');",

    "function updateUI(){",
    "  document.getElementById('tb-seg').innerHTML = DATA.segments.map(s => {",
    "    const fb = fallbackMode && s.id==='SEG-C1';",
    "    const col = fb ? '#c084fc' : (s.level==='HIGH'?'#f87171':(s.level==='MODERATE'?'#fbbf24':'#34d399'));",
    "    return '<tr><td><b>'+s.id+'</b></td><td>'+s.veh+'</td><td>'+s.speed+' km/h</td><td>'+s.queue+'m</td><td style=\"color:'+col+';font-weight:700;\">'+(fb?'KALMAN FALLBACK':s.level)+'</td></tr>';",
    "  }).join('');",
    "  document.getElementById('tb-sig').innerHTML = sigs.map(s => {",
    "    const mm = s.intended !== s.observed;",
    "    return '<tr><td><b>'+s.id+'</b></td><td style=\"color:'+(s.intended==='GREEN'?'#34d399':'#f87171')+'\">'+s.intended+'</td><td style=\"color:'+(s.observed==='GREEN'?'#34d399':'#f87171')+'\">'+s.observed+'</td><td style=\"color:'+(mm?'#ef4444':'#38bdf8')+';font-weight:700;\">'+(mm?'⚠️ MISMATCH':'SYNC OK')+'</td></tr>';",
    "  }).join('');",
    "  document.getElementById('cls-box').innerHTML = Object.keys(clsMap).map(k => '<div style=\"background:#1e293b;padding:3px 5px;border-radius:3px;display:flex;justify-content:space-between;\"><span>'+k+'</span><b>'+clsMap[k]+'</b></div>').join('');",
    "}",
    "updateUI();",

    "window.setCam = function(m, btn){",
    "  if(m===3){ showPlume = !showPlume; if(btn)btn.classList.toggle('act'); return; }",
    "  document.querySelectorAll('.cbtn').forEach((b,i)=>{ if(i<3)b.classList.remove('act'); }); if(btn)btn.classList.add('act');",
    "  if(m===0){ yaw=0.68; pitch=0.55; zoom=4.2; }",
    "  if(m===1){ yaw=0.0; pitch=1.38; zoom=3.8; }",
    "  if(m===2){ yaw=0.15; pitch=0.35; zoom=5.6; }",
    "};",

    "window.setScen = function(n, btn){",
    "  document.querySelectorAll('.sbtn').forEach(b=>b.classList.remove('act')); if(btn)btn.classList.add('act');",
    "  const sb = document.getElementById('sbanner'), pb = document.getElementById('pbadge');",
    "  if(n===1){ gcActive=false; stuckMode=false; fallbackMode=false; sigs[1].intended='RED'; sigs[1].observed='RED'; sb.innerText='Scenario 1: Normal 6-Class CV Detection & Tracking'; addLog('SCENARIO 1','Normal multi-class CV tracking active.','#38bdf8'); }",
    "  if(n===2){ gcActive=true; stuckMode=false; vehs[0].z=-82; sigs[0].observed='GREEN'; sigs[1].intended='GREEN'; sigs[1].observed='GREEN'; sb.innerText='Scenario 2: Ambulance Approaching — Dynamic 3D Green Corridor Activated'; addLog('GREEN CORRIDOR','All-Red 2.0s safety buffer verified -> Signals GREEN for AMB-101.','#10b981'); }",
    "  if(n===3){ gcActive=true; stuckMode=true; stopTimer=0; blkVerified=false; clrTimer=0; vehs[0].z=-34; vehs.filter(v=>v.blk).forEach((v,i)=>{v.z=-22+i*5.5;}); sigs[1].intended='RED'; sigs[1].observed='RED'; sb.innerText='Scenario 3: Ambulance Stuck — 2-Stage Stoppage Verification & Priority Flush'; addLog('STOPPAGE DETECTED','AMB-101 speed < 5 km/h. Verifying downstream queue via Drone CV...','#f59e0b'); }",
    "  if(n===4){ sigs[2].intended='GREEN'; sigs[2].observed='RED'; sb.innerText='Scenario 4: Signal Failure Alert at INT-SOUTH (Ctrl GREEN != Drone RED)'; addLog('SIGNAL FAULT','INT-SOUTH lamp failure detected by Drone DRN-BETA.','#ef4444'); }",
    "  if(n===5){ setCam(1); sb.innerText='Scenario 5: Autonomous Drone Aerial CV Queue & Signal Monitoring'; addLog('DRONE CV','DRN-ALPHA & DRN-BETA scanning queues (SEG-C1: 115m).','#38bdf8'); }",
    "  if(n===6){ sb.innerText='Scenario 6: Daily Volume Analytics (Weekday vs Weekend)'; askBot('Show daily traffic volume and vehicle categories today.'); }",
    "  if(n===7){ showPlume=true; sb.innerText='Scenario 7: 3D Carbon Emission Plume & Idling Fuel Savings'; askBot('How much fuel and CO2 were saved today?'); }",
    "  if(n===8){ sb.innerText='Scenario 8: GenAI Copilot Live Grounded Query'; askBot('How many ambulances passed through the green corridor today?'); }",
    "  if(n===9){ fallbackMode=true; pb.innerText='⚠️ FALLBACK: SEG-C1 CCTV OFFLINE -> DRONE + KALMAN'; pb.style.color='#fbbf24'; sb.innerText='Scenario 9: Camera Outage — Seamless Fallback to Drone CV & Kalman Filter'; addLog('SENSOR OUTAGE','SEG-C1 CCTV offline; switched to Kalman + Drone fallback.','#f59e0b'); }",
    "  if(n===10){ sb.innerText='Scenario 10: Daily Performance & Incident Summary Report'; askBot('Generate full daily traffic performance report.'); }",
    "  updateUI();",
    "};",

    "window.askBot = function(q){",
    "  const cb = document.getElementById('chat'), um = document.createElement('div');",
    "  um.className='m u'; um.innerText=q; cb.appendChild(um);",
    "  const ql = q.toLowerCase(), ts = new Date().toLocaleTimeString(); let r = '';",
    "  if(ql.includes('force')||ql.includes('change signal')||ql.includes('override')) r = '🛑 <b>[RBAC SAFETY LOCK | '+ts+']</b> Direct signal actuation via chat is blocked. Hardware TCC auth + 2.0s All-Red interlock required.';",
    "  else if(ql.includes('ambulance')&&!ql.includes('stuck')) r = '📡 <b>[Live SQLite | '+ts+']</b> <b>'+emPassed+'/14</b> emergency vehicles passed today (<b>92.9% success</b>). Avg activation: <b>1.82s</b> | Path clearance: <b>16.3s</b>.';",
    "  else if(ql.includes('stuck')) r = '📡 <b>[Live Telemetry | '+ts+']</b> '+(stuckMode?'⚠️ <b>YES:</b> AMB-101 stoppage verified on SEG-C1 ('+stopTimer.toFixed(1)+'s). Upstream flush active!':'✅ <b>NO:</b> AMB-101 moving freely at 54.5 km/h on Green Corridor.');",
    "  else if(ql.includes('congestion')||ql.includes('road')) r = '📡 <b>[Live Telemetry | '+ts+']</b> Highest congestion: <b>SEG-C1 (Central Civic Junction)</b> at <b>45.3 veh/km</b>, queue <b>115m</b>, speed <b>19.4 km/h</b>.';",
    "  else if(ql.includes('fuel')||ql.includes('co2')) r = '🌱 <b>[Model Estimate (ARAI/IPCC) | '+ts+']</b> Optimized CO2: <b>'+co2Tot.toFixed(1)+' kg</b> (<b>-'+co2Saved.toFixed(1)+' kg saved</b>). Fuel saved: <b>'+fuelSaved.toFixed(1)+' L</b>.';",
    "  else if(ql.includes('signal')||ql.includes('problem')) r = '🚦 <b>[Drone CV Audit | '+ts+']</b> Fault at <b>INT-SOUTH</b>: Controller intended <b>GREEN</b>, Drone CV observed stuck <b>RED</b>.';",
    "  else r = '📊 <b>[Daily Report | '+ts+']</b> Entered: <b>20,330</b> | Exited: <b>20,187</b> | Corridor Success: <b>92.9%</b> | Path Clr: <b>16.3s</b> | Full Queue Clr: <b>54.7s</b> | Saved: <b>'+co2Saved.toFixed(1)+' kg CO2 & '+fuelSaved.toFixed(1)+' L fuel</b>.';",
    "  const bm = document.createElement('div'); bm.className='m b'; bm.innerHTML=r; cb.appendChild(bm); cb.scrollTop=cb.scrollHeight;",
    "};",
    "window.sendChat = function(){ const i=document.getElementById('cinp'); if(i.value.trim()){ askBot(i.value.trim()); i.value=''; } };",

    "function drawPlumeSurface(){",
    "  for(let gx=-40; gx<=40; gx+=10){",
    "    ctx.beginPath();",
    "    for(let gz=-80; gz<=80; gz+=8){",
    "      const h = 22*Math.exp(-(gx*gx + gz*gz)/450) + 14*Math.exp(-(gx*gx + (gz-38)*(gz-38))/380);",
    "      const p = proj(gx, 2 + h, gz);",
    "      if(gz===-80) ctx.moveTo(p.x, p.y); else ctx.lineTo(p.x, p.y);",
    "    }",
    "    ctx.strokeStyle = 'rgba(249,115,22,0.45)'; ctx.lineWidth = 1.2; ctx.stroke();",
    "  }",
    "}",

    "function loop(){",
    "  requestAnimationFrame(loop); t += 0.016;",
    "  document.getElementById('clk').innerText = new Date().toLocaleTimeString() + ' IST';",
    "  ctx.fillStyle = '#04070e'; ctx.fillRect(0, 0, cv.width, cv.height);",

    "  for(let g=-90; g<=90; g+=15){",
    "    const p1=proj(g,0,-90), p2=proj(g,0,90), p3=proj(-90,0,g), p4=proj(90,0,g);",
    "    ctx.strokeStyle='#0f172a'; ctx.lineWidth=1;",
    "    ctx.beginPath(); ctx.moveTo(p1.x,p1.y); ctx.lineTo(p2.x,p2.y); ctx.moveTo(p3.x,p3.y); ctx.lineTo(p4.x,p4.y); ctx.stroke();",
    "  }",

    "  drawBox3D(0, -0.2, 0, 14, 0.2, 190, '#1e293b', '#0f172a', 'rgba(148,163,184,0.2)');",
    "  [-38, 0, 38].forEach(cz => drawBox3D(0, -0.18, cz, 180, 0.2, 12, '#1e293b', '#0f172a', 'rgba(148,163,184,0.2)'));",

    "  if(gcActive){",
    "    const gcCol = (stuckMode && !blkVerified) ? 'rgba(245,158,11,0.42)' : 'rgba(16,185,129,'+(0.28+0.18*Math.sin(t*5))+')';",
    "    drawBox3D(-3.5, 0.05, 0, 5.2, 0.1, 185, gcCol, gcCol, '#10b981');",
    "  }",

    "  if(showPlume) drawPlumeSurface();",
    "  bldgs.forEach((b, idx) => drawBox3D(b[0], 0, b[1], b[2], b[3], b[4], idx===5?'#1e1b4b':'#0f172a', '#1e293b', idx===5?'rgba(239,68,68,0.5)':'rgba(56,189,248,0.25)'));",

    "  if(stuckMode){",
    "    stopTimer += 0.016;",
    "    if(stopTimer < 2.4){",
    "      document.getElementById('amb-st').innerText = 'STOPPED — Verifying via Drone CV...';",
    "      document.getElementById('amb-sp').innerText = '0.0 km/h';",
    "      document.getElementById('amb-vf').innerText = 'Stage 1 ('+stopTimer.toFixed(1)+'s / 2.4s): Checking Queue';",
    "    } else if(stopTimer < 5.2){",
    "      if(!blkVerified){ blkVerified=true; sigs[1].intended='GREEN'; sigs[1].observed='GREEN'; addLog('BLOCKAGE VERIFIED (2.4s)','Queue confirmed at INT-CENTER -> Flushing GREEN!','#ef4444'); updateUI(); }",
    "      clrTimer += 0.016;",
    "      document.getElementById('amb-st').innerText = 'BLOCKAGE VERIFIED — Flushing Path!';",
    "      document.getElementById('amb-sp').innerText = '22.8 km/h';",
    "      document.getElementById('amb-vf').innerText = 'Clearing Queue ('+clrTimer.toFixed(1)+'s)';",
    "    } else {",
    "      stuckMode=false; emPassed++; document.getElementById('kv-pass').innerText = emPassed+' / 14';",
    "      document.getElementById('amb-st').innerText = 'Path Cleared — Green Corridor Active';",
    "      document.getElementById('amb-sp').innerText = '55.4 km/h';",
    "      document.getElementById('amb-vf').innerText = 'Resolved (Speed > 45 km/h)';",
    "      addLog('INCIDENT RESOLVED','AMB-101 path cleared; full speed restored.','#10b981');",
    "    }",
    "  }",

    "  vehs.forEach((v, idx) => {",
    "    const holdRed = (!v.blk && idx!==0 && v.vz>0 && sigs[1].observed==='RED' && Math.abs(v.z - (-10)) < 4);",
    "    const holdStuck = (stuckMode && stopTimer < 2.4 && (idx===0 || v.blk));",
    "    const holdCross = (gcActive && v.vx!==0 && Math.abs(v.x) < 14 && Math.abs(v.x) > 7);",
    "    if(!holdRed && !holdStuck && !holdCross){",
    "      v.x += v.vx; v.z += v.vz;",
    "      if(v.z > 92){ v.z = -92; clsMap[v.cls]++; }",
    "      if(v.z < -92){ v.z = 92; clsMap[v.cls]++; }",
    "      if(v.x > 88) v.x = -88; if(v.x < -88) v.x = 88;",
    "    }",
    "    const topPt = drawBox3D(v.x, 0.2, v.z, v.w, v.h, v.l, v.c1, v.c2, idx===0?'#ef4444':'#10b981');",
    "    if(idx===0){",
    "      const fl = Math.sin(t*18) > 0;",
    "      ctx.fillStyle = fl ? '#ef4444' : '#3b82f6';",
    "      ctx.beginPath(); ctx.arc(topPt.x, topPt.y-4, 5, 0, Math.PI*2); ctx.fill();",
    "      ctx.fillStyle = '#fff'; ctx.font = 'bold 10px sans-serif'; ctx.fillText('🚑 AMB-101', topPt.x-24, topPt.y-10);",
    "    }",
    "  });",

    "  [[-9,-38,sigs[0]], [-9,0,sigs[1]], [-9,38,sigs[2]], [28,-8,sigs[3]]].forEach(sg => {",
    "    const top = drawBox3D(sg[0], 0, sg[1], 1.2, 8.5, 1.2, '#475569', '#334155', '#64748b');",
    "    const col = sg[2].observed==='GREEN' ? '#10b981' : '#ef4444';",
    "    ctx.fillStyle = col; ctx.beginPath(); ctx.arc(top.x, top.y-3, 5.5, 0, Math.PI*2); ctx.fill();",
    "    if(sg[2].intended !== sg[2].observed){",
    "      ctx.strokeStyle = '#ef4444'; ctx.lineWidth = 2;",
    "      ctx.beginPath(); ctx.arc(top.x, top.y-3, 9 + 3*Math.sin(t*8), 0, Math.PI*2); ctx.stroke();",
    "    }",
    "  });",

    "  [[-25,'DRN-ALPHA'], [22,'DRN-BETA']].forEach((dr, i) => {",
    "    const dx = Math.cos(t*0.9 + i*3.14)*14, dz = dr[0] + Math.sin(t*0.9 + i*3.14)*12;",
    "    const dp = proj(dx, 24, dz), gp = proj(dx, 0, dz);",
    "    ctx.fillStyle = 'rgba(56,189,248,0.14)';",
    "    ctx.beginPath(); ctx.moveTo(dp.x, dp.y); ctx.lineTo(gp.x-28, gp.y); ctx.lineTo(gp.x+28, gp.y); ctx.closePath(); ctx.fill();",
    "    drawBox3D(dx, 24, dz, 3.2, 0.8, 3.2, '#38bdf8', '#0284c7', '#e0f2fe');",
    "    ctx.fillStyle = '#7dd3fc'; ctx.font = '10px sans-serif'; ctx.fillText('🛸 '+dr[1], dp.x-26, dp.y-8);",
    "  });",

    "  if(Math.floor(t*60)%90===0){",
    "    co2Saved += 0.04; co2Tot += 0.1; fuelSaved += 0.02;",
    "    document.getElementById('kv-co2').innerText = '-'+co2Saved.toFixed(1)+' kg';",
    "    document.getElementById('kv-tot').innerText = co2Tot.toFixed(1)+' kg';",
    "    document.getElementById('kv-fuel').innerText = fuelSaved.toFixed(1)+' L';",
    "    updateUI();",
    "  }",
    "}",
    "loop();",
    "</script></body></html>"
]

full_html = "\n".join(ui_lines)

from IPython.display import IFrame, display
from pathlib import Path

html_file = Path("smart_traffic_dashboard.html")
html_file.write_text(full_html, encoding="utf-8")

display(IFrame(src=str(html_file.resolve()), width="100%", height=730))